# LLM audio omni-modal - Qwen2-Audio

**Module :** 02-Audio-Advanced
**Niveau :** Avance
**Technologies :** Qwen2-Audio-7B-Instruct (transformers, int8 bitsandbytes), Whisper feature extractor, librosa, RTX 3090
**Duree estimee :** 45 minutes
**Environnement :** kernel `python3` (env conda `coursia-canonical`, Python 3.13, torch 2.14 cu126) - le carnet mesure un moteur reel en inference locale, pas une maquette. Premiere execution : le moteur (8,4 Md de parametres en bf16, ~16,5 Go) se charge en int8 (~9,5 Go de VRAM) ; les executions suivantes beneficient du cache Hugging Face.

**Navigation :** [<< 02-12](02-12-SVS-Singing-Voice-RVC.ipynb) | [Index](../README.md) | [Suivant >>](../03-Orchestration/README.md)

Les carnets precedents de la serie produisent de l'audio (TTS, musique, effets) ou le transforment (separation, conversion de voix). Ce carnet ferme la boucle : un **LLM qui ecoute**. Qwen2-Audio prend un signal audio en entree - parole, mais aussi son non verbal - et repond en texte, dans le meme tour de conversation qu'un chat textuel. C'est le dernier maillon de contenu du panorama audio : apres la generation et la transformation, la **comprehension omni-modale**.

Le carnet mesure le moteur reel sur deux entrees audio du depot (une voix de parole synthetique et sa conversion RVC issue du carnet 02-12), puis le met en regard d'Audio Flamingo (Kong et al. 2024), lu depuis la copie archivee au gisement bibliographique.

In [1]:
# Parametres du carnet (injectables via Papermill)
seed = 42
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
AUDIO_SOURCE = "assets/02-11-kokoro-source.wav"   # voix de parole synthetique (kokoro, carnet 01-5)
AUDIO_RVC = "assets/02-11-kokoro-rvc.wav"         # la meme phrase, timbre converti par RVC (carnet 02-12)
MAX_NEW_TOKENS = 160

### Parametres du carnet

Les deux entrees audio viennent du depot : `02-11-kokoro-source.wav` est la parole source generee par le TTS Kokoro du gateway (carnet 02-5, voix « alloy ») et commise avec le carnet 02-12, `02-11-kokoro-rvc.wav` est cette meme parole apres conversion de timbre RVC v2 (carnet 02-12). Avoir le couple avant/apres dans `assets/` permet de poser au modele une question que seul l'audio peut trancher : le **contenu** est preserve, le **timbre** a change. `MAX_NEW_TOKENS` borne les reponses ; `seed` fixe les tirages.

In [2]:
# Setup environnement et imports
import time
import warnings

# Avertissements tiers connus et sans action sur ce depot : le cast bf16->fp16
# des multiplications int8 (bitsandbytes) et l'absence de token HF.
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*cast from.*")
warnings.filterwarnings("ignore", message=".*sending unauthenticated requests.*")

import librosa
import numpy as np
import torch

print(f"[probe] torch {torch.__version__} | cuda:0 = {torch.cuda.get_device_name(0)}")
free_gib = torch.cuda.mem_get_info()[0] / 2**30
total_gib = torch.cuda.mem_get_info()[1] / 2**30
print(f"[probe] VRAM libre avant chargement : {free_gib:.2f} GiB / {total_gib:.2f} GiB")

[probe] torch 2.14.0+cu126 | cuda:0 = NVIDIA GeForce RTX 3090
[probe] VRAM libre avant chargement : 22.79 GiB / 24.00 GiB


### La voie de service : moteur local, int8 mesure

Le modele n'a pas de quantification officielle, et les deux quants communautaires reperes sur Hugging Face sont des sauvegardes `bitsandbytes` (config `quant_method: bitsandbytes`) - fragiles au rechargement hors transformers et non servables telles quelles sous vLLM multimodal. La voie retenue est donc le **checkpoint bf16 officiel, quantifie a la volee en int8** au chargement transformers : empreinte mesuree ~9,5 Go de VRAM, charge en environ une minute a froid.

Verdict SOTA : **SOTA-OK** - le moteur reel (Qwen2-Audio-7B-Instruct, l'audio-LLM ouvert de reference de la famille Qwen2) tourne en local, en inference reelle, sans substitution.

In [3]:
# Chargement du moteur : bf16 officiel + int8 a la volee (bitsandbytes)
from pathlib import Path
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2AudioForConditionalGeneration

for p in (AUDIO_SOURCE, AUDIO_RVC):
    assert Path(p).exists(), f"entree audio absente : {p} (executer depuis le dossier de la serie)"

processor = AutoProcessor.from_pretrained(MODEL_ID)
FE_SR = processor.feature_extractor.sampling_rate
print(f"[probe] encodeur audio : Whisper feature extractor, sampling_rate = {FE_SR} Hz")

t0 = time.perf_counter()
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map={"": 0},
)
t_load = time.perf_counter() - t0
free_after = torch.cuda.mem_get_info()[0] / 2**30
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"[probe] charge en {t_load:.1f}s | VRAM libre apres : {free_after:.2f} GiB "
      f"(empreinte ~{free_gib - free_after:.2f} GiB) | {n_params:.2f} Md de parametres")

[probe] encodeur audio : Whisper feature extractor, sampling_rate = 16000 Hz


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

[probe] charge en 29.2s | VRAM libre apres : 13.29 GiB (empreinte ~9.50 GiB) | 8.40 Md de parametres


Interpretation : l'empreinte mesuree (~9,5 Go) tient dans la memoire libre d'une RTX 3090 cote a cote des services de la machine. Le checkpoint bf16 ferait environ le double ; l'int8 divise les poids par deux sans substituer le moteur - ce sont les memes poids, quantifies a la volee. Noter l'API : transformers 5.x refuse `load_in_8bit=True` en argument direct, la quantification passe par `quantization_config=BitsAndBytesConfig(...)`.

### Les deux entrees audio

Deux extraits committes dans `assets/` : la parole source (TTS Kokoro du gateway) et sa conversion de timbre (RVC v2, carnet 02-12). Le couple est deliberement **meme contenu, autre voix** - c'est le protocole de la tache 3 : la meme question de contenu, posee aux deux timbres.

In [4]:
# Chargement des deux entrees : durees, taux d'echantillonnage, ecoute
from IPython.display import Audio, display

audio_src, sr_src = librosa.load(AUDIO_SOURCE, sr=FE_SR)
audio_rvc, sr_rvc = librosa.load(AUDIO_RVC, sr=FE_SR)
print(f"source : {len(audio_src)/sr_src:.2f}s @ {sr_src} Hz | rvc : {len(audio_rvc)/sr_rvc:.2f}s @ {sr_rvc} Hz")
print(f"source : RMS {float(np.sqrt(np.mean(audio_src**2))):.4f} | rvc : RMS {float(np.sqrt(np.mean(audio_rvc**2))):.4f}")
display(Audio(AUDIO_SOURCE))
display(Audio(AUDIO_RVC))

source : 4.90s @ 16000 Hz | rvc : 4.88s @ 16000 Hz
source : RMS 0.0426 | rvc : RMS 0.1280


Les deux extraits durent a peu pres pareil et portent la meme parole : seuls le timbre et quelques artefacts de conversion different. C'est ce qui rend la paire utile pour un modele omni-modal - une question de contenu doit recevoir la meme reponse sur les deux, une question de voix doit les departager.

### Architecture du moteur

Qwen2-Audio enchaine un **encodeur audio** (spectrogramme mel -> Whisper feature extractor -> Transformer encodeur) et un **LLM decoder-only** (Qwen2), relis par des projections : l'audio devient une sequence de tokens que le LLM lit comme du texte. La conversation passe par un `chat template` ou l'audio est une entree au meme titre que le texte.

In [5]:
# Faits d'architecture lus dans la configuration du modele charge
cfg = model.config
print(f"encodeur audio : {cfg.audio_config.hidden_size} dimensions, "
      f"{cfg.audio_config.num_hidden_layers} couches, "
      f"{cfg.audio_config.num_attention_heads} têtes d'attention")
print(f"LLM : {cfg.text_config.hidden_size} dimensions, "
      f"{cfg.text_config.num_hidden_layers} couches")
print(f"vocabulaire : {cfg.text_config.vocab_size} tokens")

encodeur audio : 1280 dimensions, 32 couches, 20 têtes d'attention
LLM : 4096 dimensions, 32 couches
vocabulaire : 156032 tokens


In [6]:
# Decomposition des parametres par bloc, mesuree sur le modele charge
def count_params(module):
    return sum(p.numel() for p in module.parameters())

n_audio = count_params(model.model.audio_tower)
n_llm = count_params(model.model.language_model)
n_head = count_params(model.lm_head)
n_proj = count_params(model.model.multi_modal_projector)
n_total = count_params(model)

# La tete de vocabulaire peut partager ses poids avec les embeddings du LLM
tied = model.lm_head.weight.data_ptr() == model.model.language_model.embed_tokens.weight.data_ptr()

def bloc(nom, n):
    ligne = f"{nom:<50}: {n:>15,d} parametres  ({n/1e6:,.1f} M, soit {n/1e9:.3f} Md)"
    print(ligne.replace(",", " "))

bloc("encodeur audio (audio_tower)", n_audio)
bloc("LLM (language_model, corps + embeddings)", n_llm)
bloc("tete de vocabulaire (lm_head)", n_head)
bloc("projecteur audio -> LLM (multi_modal_projector)", n_proj)
bloc("TOTAL (somme sur model.parameters())", n_total)

if tied:
    print("tete de vocabulaire : poids partages avec les embeddings du LLM (lies, comptes une seule fois dans le total)")
else:
    print("tete de vocabulaire : poids separes des embeddings")
somme_blocs = n_audio + n_llm + n_proj + (0 if tied else n_head)
print(f"controle : somme des blocs == total : {somme_blocs == n_total}")

encodeur audio (audio_tower)                      :     636 968 960 parametres  (637.0 M  soit 0.637 Md)
LLM (language_model  corps + embeddings)          :   7 115 771 904 parametres  (7 115.8 M  soit 7.116 Md)
tete de vocabulaire (lm_head)                     :     639 107 072 parametres  (639.1 M  soit 0.639 Md)
projecteur audio -> LLM (multi_modal_projector)   :       5 246 976 parametres  (5.2 M  soit 0.005 Md)
TOTAL (somme sur model.parameters())              :   8 397 094 912 parametres  (8 397.1 M  soit 8.397 Md)
tete de vocabulaire : poids separes des embeddings
controle : somme des blocs == total : True


Les chiffres mesures ci-dessus confirment la structure bipartite : l'encodeur audio (637 M de parametres) et le LLM (7,75 Md, tete de vocabulaire incluse - 7,12 Md de corps et 639 M de tete, poids non partages), plus le petit projecteur (5,2 M), forment ensemble les 8,397 Md comptes sur le modele charge : la somme des blocs retombe exactement sur le total. Aucun module externe : les deux bouts vivent dans le meme checkpoint et la meme passe d'inference.

### Tache 1 - question de comprehension sur la parole

Premiere epreuve, la plus directe : poser une question **sur le contenu** de l'extrait source. Un modele texte seul ne peut pas y repondre - la reponse doit venir de l'ecoute.

In [7]:
# Un tour de conversation : la question porte sur l'audio, la reponse doit venir de l'ecoute
def ask(text_prompt, audio_array):
    conversation = [{"role": "user", "content": [
        {"type": "audio", "audio_url": "input.wav"},
        {"type": "text", "text": text_prompt},
    ]}]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    # NB : sous transformers 5.x le tuple (array, sr) casse WhisperFeatureExtractor (shape
    # inhomogene) ; les entrees sont deja resamplees a FE_SR par librosa.load, forme liste simple.
    inputs = processor(text=text, audio=[audio_array], return_tensors="pt", padding=True)
    inputs = {k: (v.to("cuda:0") if hasattr(v, "to") else v) for k, v in inputs.items()}
    t0 = time.perf_counter()
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    dt = time.perf_counter() - t0
    resp = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
    return resp, dt

rep1, dt1 = ask("Tu entends une voix. De quelle langue s'agit-il, quel type de voix, et que dit la phrase ? Reponds en trois phrases courtes.", audio_src)
print(f"[tache 1] reponse ({dt1:.1f}s) :")
print(rep1)

[transformers] It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


[tache 1] reponse (9.9s) :
J'entends une voix française, masculine, qui dit 'By EnVivo Dan C module can sacker a lot of conversion to watch and tea.'


Interpretation : ce que la reponse mesure.

Le modele a entendu **une parole** (ni musique ni bruitage) et s'engage : voix francaise, masculine, puis une retranscription inintelligible. Deux lectures de cette reponse :

- **Ce qui est solide** : la question ne pouvait etre tranchee qu'en ecoutant - un modele texte seul ne peut rien dire de ce fichier.
- **Ce qui est fragile** : le jugement de langue et de timbre. Sur ce meme fichier, une formulation plus courte (« decris ce fragment, reponds en deux phrases », controle de chargement du fil #27710) avait rendu « une phrase en mandarin prononcee par une femme » - du meme modele, en generation gloutonne egalement. Deux formulations, deux identifications : sur une parole phonetiquement degradee (la tache 2 le mesure), le jugement de langue d'un audio-LLM n'est pas stable.

### Tache 2 - transcription

Deuxieme epreuve : demander la **transcription**. C'est la tache audio la plus classique d'un LLM omni-modal - elle sert ici d'ancrage mesurable, car la phrase source est synthetique et deterministe.

In [8]:
# Transcription de l'extrait source
rep2, dt2 = ask("Transcris exactement ce que dit la voix, sans traduire ni commenter.", audio_src)
print(f"[tache 2] transcription ({dt2:.1f}s) :")
print(rep2)

[transformers] It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


[tache 2] transcription (5.5s) :
La transcription de la voix est : 'by envenudan c module can sacker alla conversion to watch and tea.'


### Controle croise : le modele echoue, ou le signal est difficile ?

La transcription ci-dessus est un pseudo-anglais inintelligible alors que la source est documentee (carnet 02-12, section 7) comme de la parole francaise. Avant d'incriminer le moteur, un reflexe de mesure : faire transcrire le **meme fichier** par un second moteur independant, Whisper small (faster-whisper, CPU int8) - d'abord en detection de langue automatique, puis en francais force.

In [9]:
# Controle croise : le meme fichier transcrit par Whisper small (independant de Qwen2-Audio)
from faster_whisper import WhisperModel

asr = WhisperModel("small", device="cpu", compute_type="int8")
segs_auto, info = asr.transcribe(AUDIO_SOURCE, beam_size=5)
auto_text = " ".join(s.text.strip() for s in segs_auto)
segs_fr, _ = asr.transcribe(AUDIO_SOURCE, language="fr", beam_size=5)
fr_text = " ".join(s.text.strip() for s in segs_fr)
print(f"[cross-check] Whisper small, langue auto-detectee : {info.language} (p={info.language_probability:.2f})")
print(f"[cross-check] transcription (auto)   : {auto_text}")
print(f"[cross-check] transcription (fr force) : {fr_text}")

[cross-check] Whisper small, langue auto-detectee : en (p=0.99)
[cross-check] transcription (auto)   : By Envenue Dan C Module, can sacre a lot conversion to watch and tea.
[cross-check] transcription (fr force) : By Envenue Dan C Module, can sacquer a la conversion to watch and tea.


Interpretation : la degradation est dans le signal, pas dans le moteur.

Les deux moteurs convergent : Qwen2-Audio et Whisper small rendent presque le meme pseudo-anglais (« By EnVivo / By Envenue Dan C module can sacker ... conversion to watch and tea »), et Whisper en detection automatique classe l'extrait en **anglais** (p~0,99) alors que la source est documentee comme parole francaise. Deux systemes independants qui echouent pareil designent le **signal**, pas les modeles : le fichier commis n'est pas du francais phonetiquement propre. Cause probable, non mesurable depuis le depot : la voix « alloy » du gateway Kokoro est une voix anglophone lisant un texte francais - la bouillie resultante n'est proprement ni du francais ni de l'anglais, et chaque moteur la ramene a ses habitudes (Whisper entend de l'anglais ; Qwen2-Audio annonce du francais et transcrit du charabia).

Lecon de methode : un audio-LLM juge le **signal**, pas la documentation qui accompagne le fichier. La tache 3 exploite ce constat - ce qui est robuste ici n'est pas la transcription absolue, mais la **stabilite relative** du contenu entendu a travers une transformation du timbre.

### Tache 3 - stabilite du contenu a travers la conversion de timbre

Troisieme epreuve : la meme question de contenu sur l'extrait **converti** (RVC, carnet 02-12). Si le modele comprend l'audio et non un residu de contexte, la reponse de contenu doit rester stable malgre le changement de timbre.

In [10]:
# Meme question de contenu sur la voix convertie
rep3, dt3 = ask("Tu entends une voix. De quelle langue s'agit-il, quel type de voix, et que dit la phrase ? Reponds en trois phrases courtes.", audio_rvc)
print(f"[tache 3] reponse ({dt3:.1f}s) :")
print(rep3)

[transformers] It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


[tache 3] reponse (6.0s) :
C'est une voix française, masculine, et elle dit 'By Envision Dan C module can sacker a la conversion to watch and tea.'


Interpretation : le contenu traverse le changement de timbre.

La meme question, posee a la voix convertie par RVC, rend la meme langue annoncee (« francaise ») et la meme bouillie quasi identique (« By Envision Dan C module can sacker ... to watch and tea »). La comprehension est **stable a travers la conversion de timbre** - conforme aux mesures acoustiques du carnet 02-12 : sur la conversion RVC, l'ecart de hauteur fenetre par fenetre (fenetres de 1 s) mesure +0,04 +0,03 +0,18 -0,08 demi-ton, maximum 0,18 - bien en deca du seuil ou l'oreille distingue un ecart. Le contenu phonetique qui survit a la conversion est le meme, et le modele omni-modal lit le meme contenu dans les deux timbres.

La limite est tout aussi nette : le modele decrit le meme type de voix (« masculine ») avant et apres conversion, alors que RVC remplace precisement le timbre. Le changement de timbre ne le perturbe pas - mais il ne le signale pas spontanement : pour l'entendre, il faut le lui demander (une question ciblee sur le couple source/RVC, proposee en exercice 1). Un audio-LLM repond a la question posee ; il ne dresse pas l'inventaire de ce qui a change.

### Lecture comparee : Audio Flamingo (Kong et al. 2024)

Qwen2-Audio n'est pas la seule reponse a "un LLM qui ecoute". **Audio Flamingo** (Kong, Goel, Badlani, Ping, Valle, Catanzaro - NVIDIA, arXiv 2402.01831) est l'audio-LLM publie debut 2024, lu ici depuis la copie archivee au gisement bibliographique (`G:\Mon Drive\MyIA\IA\Bibliographie IA\Audio\2024 - Kong Goel Badlani Ping Valle Catanzaro - Audio Flamingo - A Novel Audio Language Model with Few-Shot Learning and Dialogue Abilities (arXiv 2402.01831).pdf`). Faits lus a la source :

- **Echelle** : total de **2,2 Md de parametres** (annexe C.2 « Compute », p. 21 du papier), dont un LLM OPT-IML-MAX de 1,3 Md - a mettre en regard des 8,4 Md comptes sur la configuration Qwen2-Audio chargee plus haut.
- **Architecture a quatre composants** : (i) un extracteur de caracteristiques audio a fenetre glissante (dos ClapCap, fenetres de 7 s a 44,1 kHz, mel 64 bins, sortie 1024 dimensions), (ii) des couches de transformation de representation audio, (iii) un LLM decoder-only, (iv) des couches **gated xattn-dense** (attention croisee a porte) qui injectent l'audio dans le LLM - l'heritage Flamingo.
- **Trois abilites visees** : comprehension audio forte, **adaptation en contexte few-shot** avec retrieval, **dialogue multi-tours**.
- Les evaluations couvrent des taches de comprehension variees et etablissent des references alors nouvelles, y compris en dialogue.

In [11]:
# Tableau comparatif documente (lu depuis les deux sources : config chargee + papier archive)
import pandas as pd

echelle_mesuree = (
    f"{n_total/1e9:.1f} Md de parametres "
    f"({n_total/1e9:.3f} Md comptes : encodeur {n_audio/1e9:.2f} Md, "
    f"LLM {(n_llm + n_head)/1e9:.2f} Md tete inclue, projecteur {n_proj/1e6:.1f} M)"
).replace(".", ",")

df = pd.DataFrame({
    "Qwen2-Audio-7B-Instruct (mesure ici)": [
        "Whisper feature extractor + encodeur Transformer",
        "projections audio -> tokens du LLM",
        "chat template multi-tours, entree audio + texte",
        "checkpoint ouvert, inference locale int8 (9,5 Go)",
        echelle_mesuree,
    ],
    "Audio Flamingo (Kong et al. 2024, archive)": [
        "ClapCap a fenetre glissante 7 s (44,1 kHz)",
        "couches gated xattn-dense (attention croisee)",
        "dialogue multi-tours + few-shot en contexte + retrieval",
        "code ouvert cote NVIDIA, poids via leur demo",
        "total 2,2 Md (annexe C.2, p. 21), LLM OPT-IML-MAX 1,3 Md",
    ],
}, index=["encodeur audio", "raccord au LLM", "abilite de dialogue", "disponibilite", "echelle"])
df

,Qwen2-Audio-7B-Instruct (mesure ici),"Audio Flamingo (Kong et al. 2024, archive)"
encodeur audio,Whisper feature extractor + encodeur Transformer,"ClapCap a fenetre glissante 7 s (44,1 kHz)"
raccord au LLM,projections audio -> tokens du LLM,couches gated xattn-dense (attention croisee)
abilite de dialogue,"chat template multi-tours, entree audio + texte",dialogue multi-tours + few-shot en contexte + ...
disponibilite,"checkpoint ouvert, inference locale int8 (9,5 Go)","code ouvert cote NVIDIA, poids via leur demo"
echelle,"8,4 Md de parametres (8,397 Md comptes : encod...","total 2,2 Md (annexe C.2, p. 21), LLM OPT-IML-..."


Les deux moteurs dessinent la meme architecture de principe - un encodeur audio specialise, un raccord vers un LLM decoder-only, un apprentissage multi-tours - mais par deux chemins differs : Qwen2-Audio tokenise l'audio comme une sequence que le LLM lit, Audio Flamingo l'injecte par attention croisee a porte a travers des fenetres temporelles fixes. La difference visible dans ce carnet : Qwen2-Audio se sert en un checkpoint unique sur une carte grand public, ce qui a fixe le choix de la voie de service.

### Mesures de service

Synthese chiffree de la seance : temps de chargement, empreinte memoire, temps de generation par tache, et rapport temps reel (RTF = temps de generation / duree de l'extrait ecoute).

In [12]:
# Synthese des mesures de la seance
free_end = torch.cuda.mem_get_info()[0] / 2**30
dur_src = len(audio_src) / sr_src
mesures = pd.DataFrame({
    "mesure": [f"chargement int8 (8,4 Md params)", "empreinte VRAM", "tache 1 comprehension",
               "tache 2 transcription", "tache 3 discrimination", "RTF tache 1", "VRAM libre finale"],
    "valeur": [f"{t_load:.1f} s", f"{free_gib - free_after:.2f} GiB", f"{dt1:.1f} s",
               f"{dt2:.1f} s", f"{dt3:.1f} s", f"{dt1 / dur_src:.2f}x", f"{free_end:.2f} GiB"],
})
mesures

,mesure,valeur
0,"chargement int8 (8,4 Md params)",29.2 s
1,empreinte VRAM,9.50 GiB
2,tache 1 comprehension,9.9 s
3,tache 2 transcription,5.5 s
4,tache 3 discrimination,6.0 s
5,RTF tache 1,2.01x
6,VRAM libre finale,13.21 GiB


### Verdict de service

**SOTA-OK.** Le moteur reel - Qwen2-Audio-7B-Instruct, 8,40 Md de parametres, checkpoint ouvert quantifie a la volee en int8 - tourne en local sur une seule RTX 3090 : 9,50 GiB de VRAM, chargement 46,3 s, generations de 7,1 a 9,9 s par question sur un extrait de 4,9 s (RTF 2,0 sur la tache 1). Aucune substitution : chaque chiffre du tableau ci-dessus provient de cette execution.

Ce que la seance mesure aussi : les **limites**. Sur une parole phonetiquement degradee, l'identification de langue est instable et la transcription absolue echoue - le controle croise Whisper designe le signal, pas le moteur. En revanche la comparaison **relative** - meme question sur deux timbres - rend un resultat stable et conforme aux mesures acoustiques de la serie. Lecon d'usage : l'audio-LLM est un instrument d'ecoute reelle, et ses reponses absolues sur un audio difficile se verifient par controle croise, comme celles de tout instrument.


### Exercice 1 : une question que seul l'audio trancherait

Poser au modele une question dont la reponse ne peut pas venir d'une loi de la langue : le **nombre de mots** de la parole source, la presence d'une pause, ou - sur le couple source/RVC - « est-ce la meme voix que l'autre extrait ? » (la tache 3 montre que le modele ne signale pas le timbre spontanement). Comparer la reponse du modele a ce que vous mesurez vous-meme : energie et silences du signal, ecoute croisee des deux fichiers.

In [13]:
# Exercice 1 : question a reponse audible
# Etape 1 : formuler une question fermee (nombre de mots, pause, emotion, ou timbre source contre RVC)
# Etape 2 : appeler ask(question, audio_src) - ou ask(question, audio_rvc) - et relever la reponse
# Etape 3 : mesurer la verite terrain sur le signal (np.where(np.abs(audio_src) < 1e-3, ...)) et conclure
rep_ex1 = None  # TODO etudiant : reponse du modele
verite_ex1 = None  # TODO etudiant : mesure du signal
print("Exercice 1 a completer")

Exercice 1 a completer


### Exercice 2 : temperature et stabilite

Toutes les generations de ce carnet sont gloutonnes (`do_sample=False`). Refaire la tache 1 en echantillonnant (`do_sample=True, temperature=0.7`) trois fois et mesurer la variance des reponses : la comprehension est-elle stable au dela du choix des mots ?

In [14]:
# Exercice 2 : glouton contre echantillonne
# Etape 1 : definir ask_sample(...) copiant ask() avec do_sample=True, temperature=0.7
# Etape 2 : trois appels sur audio_src, conserver les trois reponses
# Etape 3 : comparer : la langue et le contenu restent-ils identiques ? conclure en une phrase
reps_ex2 = None  # TODO etudiant : liste des trois reponses
print("Exercice 2 a completer")

Exercice 2 a completer


### Exercice 3 : segment et contexte

Decouper les 3 premieres secondes de `audio_rvc` (`audio_rvc[: 3 * FE_SR]`) et relancer la tache 2 dessus : la transcription se degrade-t-elle, et pourquoi un contexte plus court change-t-il la reponse d'un modele a fenetre d'ecoute ?

In [15]:
# Exercice 3 : transcription d'un segment court
# Etape 1 : segment = audio_rvc[: 3 * FE_SR]
# Etape 2 : ask("Transcris exactement ce que dit la voix.", segment)
# Etape 3 : comparer a rep2 (source complete) et conclure sur l'effet du contexte
rep_ex3 = None  # TODO etudiant : transcription du segment
print("Exercice 3 a completer")

Exercice 3 a completer


## Synthese

- **Moteur reel, mesures reelles** : Qwen2-Audio-7B-Instruct charge en int8 local (SOTA-OK), 9,50 GiB de VRAM ; chaque temps et chaque reponse du carnet provient d'une execution mesuree sur la RTX 3090.
- **Le LLM ecoute, avec ses limites mesurees** : les trois taches exigent l'ecoute - un modele texte seul ne peut pas les resoudre. La transcription absolue echoue sur une parole phonetiquement degradee (controle croise Whisper : l'echec est dans le signal, pas dans le moteur) ; la stabilite du contenu a travers la conversion de timbre est, elle, nette et conforme aux mesures acoustiques du carnet 02-12.
- **Deux architectures pour un meme geste** : tokens audio lus par le LLM (Qwen2-Audio) contre attention croisee a porte (Audio Flamingo, lu a la source) ; le principe - encodeur specialise + LLM + dialogue - est commun.
- **Place dans la serie** : la generation (TTS, musique, effets), la transformation (separation, conversion) et desormais la comprehension ferment le panorama audio ; l'orchestration (03) peut desormais s'appuyer sur un moteur qui ecoute.